# ViVQA Comprehensive Ablation Study
## Nghiên cứu thực nghiệm hệ thống cho Luận Văn

### Quy trình 5 bước:

| Bước | Phase | File chạy | Mục đích | Đầu ra cho Paper |
|------|-------|-----------|----------|-------------------|
| 1 | Phase 1: Loss Weight Ablation | ablationstudy | Dò bộ λ tối ưu (10 epoch, subset 15%) | Bảng & Heatmap: λ\_bbox × λ\_giou |
| 2 | *(Không ở đây)* | train.py | Train full 100% data với λ tốt nhất | File model chính |
| 3 | Phase 2: Inference Strategy | ablationstudy | Gắn model chính vào, tìm pad\_ratio & crop\_weight | Bảng: Inference vs Accuracy |
| 4 | Phase 3: Structural Ablation | ablationstudy | So sánh 4 biến thể kiến trúc trên subset | Bảng & Bar chart: Module contribution |
| 5 | Phase 4: Vision Depth Ablation | ablationstudy | Thay đổi số layer QFormer unfreeze | Bảng & Line chart: Depth vs Performance |

> **Lưu ý:** Phase 3 & 4 chạy độc lập trên subset, không cần model từ Bước 2.
> Phase 2 **bắt buộc** cần model chính từ Bước 2.

In [1]:
import os
from google.colab import drive

# 1. Mount Google Drive
drive.mount('/content/drive')

# 2. Điền thông tin Kaggle API của bạn vào đây
# (Vào Kaggle -> Settings -> Create New Token để lấy username và key)
os.environ['KAGGLE_USERNAME'] = "huyphngquc"
os.environ['KAGGLE_KEY'] = "d2051dddb134d7809d29527c977cab58"

# Tạo thư mục trên Drive để lưu dataset cố định (nếu chưa có)
DRIVE_DATASET_DIR = '/content/drive/MyDrive/Thesis_ViVQA_Data'
os.makedirs(DRIVE_DATASET_DIR, exist_ok=True)

Mounted at /content/drive


In [ ]:
# # ==============================================================================
# # CHỈ CHẠY 1 LẦN: Tải dataset từ Kaggle về thẳng Google Drive
# # SAU KHI TẢI XONG, HÃY COMMENT ĐOẠN CODE NÀY LẠI
# # ==============================================================================
# !kaggle datasets download -d hydrogenhydrogen/vivqa-1 -p {DRIVE_DATASET_DIR}

# print(f"✅ Đã tải file zip về: {DRIVE_DATASET_DIR}/vivqa-1.zip")

In [3]:
import os
import time

LOCAL_DATA_DIR = '/content/vivqa_dataset'
ZIP_PATH_ON_DRIVE = f'{DRIVE_DATASET_DIR}/vivqa-1.zip' # Đảm bảo biến này đúng
ZIP_PATH_LOCAL = '/content/vivqa-1.zip'

start_time = time.time()

# 1. Xóa rác cũ nếu có (Quan trọng để tránh giải nén đè lên file lỗi)
!rm -rf {LOCAL_DATA_DIR}
!rm -f {ZIP_PATH_LOCAL}

# 2. Copy bằng lệnh Linux (Trâu hơn shutil)
print("Đang copy từ Drive sang Local...")
!cp "{ZIP_PATH_ON_DRIVE}" "{ZIP_PATH_LOCAL}"

# 3. Giải nén và TỰ ĐỘNG SỬA LỖI (Dùng option -FF nếu cần)
print("Đang giải nén...")
os.makedirs(LOCAL_DATA_DIR, exist_ok=True)

# Thử giải nén bình thường trước
result = !unzip -q -n {ZIP_PATH_LOCAL} -d {LOCAL_DATA_DIR}

# Kiểm tra nếu vẫn còn báo lỗi "bad zipfile offset"
if any("bad zipfile offset" in line for line in result):
    print("⚠️ File zip có vẻ bị lỗi offset, đang thử fix...")
    # Lệnh fix file zip của linux
    !zip -F {ZIP_PATH_LOCAL} --out {ZIP_PATH_LOCAL}_fixed.zip
    !unzip -q -n {ZIP_PATH_LOCAL}_fixed.zip -d {LOCAL_DATA_DIR}

# 4. Dọn dẹp
if os.path.exists(ZIP_PATH_LOCAL): os.remove(ZIP_PATH_LOCAL)

print(f"✅ Xong! Kiểm tra thử: {len(os.listdir(LOCAL_DATA_DIR))} folders/files.")

✅ Xong! Kiểm tra thử: 13 folders/files.


In [4]:
# 4. Kiểm tra và in ra cấu trúc thực tế
print("\n--- KIỂM TRA CẤU TRÚC DATA ---")
if os.path.exists(LOCAL_DATA_DIR):
    # Liệt kê tất cả file và folder (kể cả trong folder con)
    all_files = []
    for root, dirs, files in os.walk(LOCAL_DATA_DIR):
        for name in files:
            all_files.append(os.path.join(root, name))
    
    print(f"📍 Path cơ sở: {LOCAL_DATA_DIR}")
    print(f"📦 Tổng số lượng file tìm thấy: {len(all_files)}")
    
    if len(all_files) > 0:
        print("📄 5 file đầu tiên để check path:")
        for f in all_files[:5]:
            print(f"   -> {f}")
    else:
        print("❌ CẢNH BÁO: Thư mục tồn tại nhưng không có file nào bên trong (do giải nén lỗi).")
else:
    print("❌ LỖI: Thư mục LOCAL_DATA_DIR không tồn tại.")


--- KIỂM TRA CẤU TRÚC DATA ---
📍 Path cơ sở: /content/vivqa_dataset
📦 Tổng số lượng file tìm thấy: 11720
📄 5 file đầu tiên để check path:
   -> /content/vivqa_dataset/merged_train.csv
   -> /content/vivqa_dataset/yolo_final_results.csv
   -> /content/vivqa_dataset/answer_space.txt
   -> /content/vivqa_dataset/Owl_train.csv
   -> /content/vivqa_dataset/answer_type_mapping.csv


In [5]:
!pip install -q transformers timm accelerate bitsandbytes scikit-learn underthesea pandas tqdm tabulate matplotlib seaborn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 88.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 59.1 MB/s eta 0:00:00


In [6]:
import os
import ast
import gc
import time
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
import itertools
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
from transformers import Blip2Processor, Blip2Model, AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from torch.cuda.amp import autocast, GradScaler
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from underthesea import word_tokenize

import warnings
warnings.filterwarnings("ignore")

# Cấu hình đồ thị cho Paper
plt.rcParams['figure.dpi'] = 150
plt.rcParams['font.size'] = 10
sns.set_style("whitegrid")

In [7]:
import warnings, logging, sys
warnings.filterwarnings("ignore", category=UserWarning)
logging.getLogger("torch.utils.data.dataloader").setLevel(logging.ERROR)

if not hasattr(sys, '_custom_unraisablehook_applied'):
    original_hook = sys.unraisablehook
    def filter_dataloader_del_errors(unraisable):
        if issubclass(unraisable.exc_type, AssertionError) and "can only test a child process" in str(unraisable.exc_value):
            pass
        else:
            original_hook(unraisable)
    sys.unraisablehook = filter_dataloader_del_errors
    sys._custom_unraisablehook_applied = True

In [8]:
# ==========================================
# CẤU HÌNH TỔNG THỂ CHO 4 PHASE ABLATION
# ==========================================
CONFIG = {
    # ============ DATA PATHS ============
    "train_csv": "/content/vivqa_dataset/ViVQA-main/ViVQA-main/train.csv",
    "train_img_dir": "/content/vivqa_dataset/train",
    "train_box_csv": "/content/vivqa_dataset/merged_train.csv",

    # ============ MODEL ============
    "blip_model": "Salesforce/blip2-opt-2.7b",
    "text_model": "vinai/phobert-base",
    "device": "cuda" if torch.cuda.is_available() else "cpu",
    "batch_size": 32,
    "type_embed_dim": 64,
    "max_length": 50,
    "box_required_types": [2],
    "qformer_unfreeze_layers": 2,  # Mặc định, Phase 4 sẽ thay đổi

    # ============ [PHASE 1] Loss Weight Grid ============
    "dev_set_ratio": 0.15,
    "val_set_ratio": 0.05,
    "phase1_epochs": 10,
    "grid_training": {
        "lambda_bbox": [0.5, 1.0, 2.0, 5.0],
        "lambda_giou": [0.5, 1.0, 5.0, 10.0],
    },
    "lambda_qtype": 1.0,  # Cố định (ít ảnh hưởng)

    # ============ [PHASE 2] Inference Grid ============
    "grid_inference": {
        "pad_ratio": [0.0, 0.05, 0.10, 0.15, 0.25],
        "crop_weight": [0.3, 0.4, 0.5, 0.6, 0.7, 0.8],
    },

    # ============ [PHASE 3] Structural ============
    "phase3_epochs": 10,

    # ============ [PHASE 4] QFormer Depth Grid ============
    "phase4_epochs": 10,
    "grid_qformer_layers": [0, 1, 2, 3, 4, 6],

    # ============ OUTPUT PATHS ============
    "phase1_result_csv": "/content/drive/MyDrive/ablation_phase1_loss_weights.csv",
    "phase2_result_csv": "/content/drive/MyDrive/ablation_phase2_inference.csv",
    "phase3_result_csv": "/content/drive/MyDrive/ablation_phase3_structural.csv",
    "phase4_result_csv": "/content/drive/MyDrive/ablation_phase4_depth.csv",
    "checkpoint_to_test": "/content/drive/MyDrive/vivqa_v100_100percent.pth",
    "figures_dir": "/content/drive/MyDrive/ablation_figures",
}

os.makedirs(CONFIG['figures_dir'], exist_ok=True)
print(f"Device: {CONFIG['device']}")
n_p1 = len(list(itertools.product(*CONFIG['grid_training'].values())))
n_p2 = len(list(itertools.product(*CONFIG['grid_inference'].values())))
print(f"Phase 1: {n_p1} tổ hợp × {CONFIG['phase1_epochs']} epochs")
print(f"Phase 2: {n_p2} tổ hợp (inference only, không train)")
print(f"Phase 3: 4 kiến trúc × {CONFIG['phase3_epochs']} epochs")
print(f"Phase 4: {len(CONFIG['grid_qformer_layers'])} mức depth × {CONFIG['phase4_epochs']} epochs")

Device: cuda
Phase 1: 16 tổ hợp × 10 epochs
Phase 2: 30 tổ hợp (inference only, không train)
Phase 3: 4 kiến trúc × 10 epochs
Phase 4: 6 mức depth × 10 epochs


In [9]:
# ==========================================
# HÀM BỔ TRỢ VÀ METRICS
# ==========================================
def preprocess_vietnamese(text):
    text = str(text).lower().strip()
    return word_tokenize(text, format="text")

def find_image_path(folder, img_id):
    valid_exts = ['.jpg', '.png', '.jpeg']
    img_id_str = str(img_id)
    if img_id_str.lower().endswith(tuple(valid_exts)):
        path = os.path.join(folder, img_id_str)
        if os.path.exists(path): return path
    for ext in valid_exts:
        path = os.path.join(folder, f"{img_id_str}{ext}")
        if os.path.exists(path): return path
    return None

def calculate_iou(pred_boxes, target_boxes):
    p_left = torch.min(pred_boxes[:, 0], pred_boxes[:, 2])
    p_top = torch.min(pred_boxes[:, 1], pred_boxes[:, 3])
    p_right = torch.max(pred_boxes[:, 0], pred_boxes[:, 2])
    p_bottom = torch.max(pred_boxes[:, 1], pred_boxes[:, 3])

    inter_xmin = torch.max(p_left, target_boxes[:, 0])
    inter_ymin = torch.max(p_top, target_boxes[:, 1])
    inter_xmax = torch.min(p_right, target_boxes[:, 2])
    inter_ymax = torch.min(p_bottom, target_boxes[:, 3])

    inter_w = torch.clamp(inter_xmax - inter_xmin, min=0)
    inter_h = torch.clamp(inter_ymax - inter_ymin, min=0)
    inter_area = inter_w * inter_h

    pred_area = (p_right - p_left) * (p_bottom - p_top)
    target_area = (target_boxes[:, 2] - target_boxes[:, 0]) * (target_boxes[:, 3] - target_boxes[:, 1])
    union_area = pred_area + target_area - inter_area + 1e-6

    iou = inter_area / union_area
    return torch.clamp(iou, min=0.0, max=1.0)

def giou_loss(pred_boxes, target_boxes):
    px1, py1, px2, py2 = pred_boxes[:, 0], pred_boxes[:, 1], pred_boxes[:, 2], pred_boxes[:, 3]
    tx1, ty1, tx2, ty2 = target_boxes[:, 0], target_boxes[:, 1], target_boxes[:, 2], target_boxes[:, 3]

    p_left, p_right = torch.min(px1, px2), torch.max(px1, px2)
    p_top, p_bottom = torch.min(py1, py2), torch.max(py1, py2)
    pred_area = (p_right - p_left + 1e-6) * (p_bottom - p_top + 1e-6)
    target_area = (tx2 - tx1) * (ty2 - ty1)

    inter_xmin, inter_ymin = torch.max(p_left, tx1), torch.max(p_top, ty1)
    inter_xmax, inter_ymax = torch.min(p_right, tx2), torch.min(p_bottom, ty2)
    inter_w = torch.clamp(inter_xmax - inter_xmin, min=0)
    inter_h = torch.clamp(inter_ymax - inter_ymin, min=0)
    inter_area = inter_w * inter_h

    union_area = pred_area + target_area - inter_area + 1e-6
    iou = inter_area / union_area

    enc_xmin, enc_ymin = torch.min(p_left, tx1), torch.min(p_top, ty1)
    enc_xmax, enc_ymax = torch.max(p_right, tx2), torch.max(p_bottom, ty2)
    enc_area = torch.clamp(enc_xmax - enc_xmin, min=0) * torch.clamp(enc_ymax - enc_ymin, min=0) + 1e-6

    giou = iou - (enc_area - union_area) / enc_area
    return (1 - giou).mean()


def compute_extended_metrics(all_preds, all_labels, all_qtypes, all_ious=None, num_q_types=None):
    """Tính toàn bộ metrics cần cho paper: Acc, F1-macro, F1-weighted, per-type accuracy, box metrics."""
    results = {}
    preds = np.array(all_preds)
    labels = np.array(all_labels)

    # Overall VQA Accuracy
    results['vqa_acc'] = float((preds == labels).mean())

    # F1 Scores
    results['f1_macro'] = float(f1_score(labels, preds, average='macro', zero_division=0))
    results['f1_weighted'] = float(f1_score(labels, preds, average='weighted', zero_division=0))

    # Per Question-Type Accuracy
    qtypes = np.array(all_qtypes)
    if num_q_types is None:
        num_q_types = int(qtypes.max()) + 1
    for t in range(num_q_types):
        mask = qtypes == t
        if mask.sum() > 0:
            results[f'acc_type_{t}'] = float((preds[mask] == labels[mask]).mean())
            results[f'count_type_{t}'] = int(mask.sum())

    # Box metrics
    if all_ious is not None and len(all_ious) > 0:
        ious = np.array(all_ious)
        results['miou'] = float(ious.mean())
        results['box_acc_05'] = float((ious >= 0.5).mean())   # Acc@IoU=0.5
        results['box_acc_075'] = float((ious >= 0.75).mean())  # Acc@IoU=0.75

    return results


def count_trainable_params(model):
    """Đếm số lượng tham số trainable."""
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

def format_number(n):
    """Format số param cho đẹp."""
    if n >= 1e6: return f"{n/1e6:.2f}M"
    if n >= 1e3: return f"{n/1e3:.1f}K"
    return str(n)

In [10]:
# ==========================================
# 3. DATASET
# ==========================================
class ViVQATripleTaskDataset(Dataset):
    def __init__(self, dataframe, image_dir, blip_processor, tokenizer, label_encoder, unk_token_id):
        self.data = dataframe
        self.image_dir = image_dir
        self.blip_processor = blip_processor
        self.tokenizer = tokenizer
        self.label_encoder = label_encoder
        self.unk_token_id = unk_token_id
        self.known_classes = set(label_encoder.classes_)

    def __len__(self): return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        img_id = row.get('img_id', row.get('image'))
        img_path = find_image_path(self.image_dir, img_id)
        try:
            image = Image.open(img_path).convert("RGB") if img_path else Image.new('RGB', (224, 224))
        except:
            image = Image.new('RGB', (224, 224))
        orig_w, orig_h = image.size
        pixel_values = self.blip_processor(images=image, return_tensors="pt")['pixel_values'].squeeze(0)

        target_box = torch.zeros(4, dtype=torch.float32)
        has_box = torch.tensor(0.0, dtype=torch.float32)
        
        # CẬP NHẬT: Ưu tiên lấy merged_box, nếu không có mới lấy owl_box
        box_str = row.get('merged_box', row.get('owl_box', None))
        
        if pd.notna(box_str):
            try:
                box = ast.literal_eval(str(box_str))
                xmin, ymin, xmax, ymax = box
                target_box[0], target_box[1] = xmin / orig_w, ymin / orig_h
                target_box[2], target_box[3] = xmax / orig_w, ymax / orig_h
                target_box = torch.clamp(target_box, 0.0, 1.0)
                has_box = torch.tensor(1.0, dtype=torch.float32)
            except:
                pass

        question = preprocess_vietnamese(row['question'])
        text_encoding = self.tokenizer(
            question, return_tensors="pt", padding="max_length",
            truncation=True, max_length=CONFIG['max_length'], add_special_tokens=True
        )
        answer = str(row.get('answer', '')).lower().strip()
        label = self.label_encoder.transform([answer])[0] if answer in self.known_classes else self.unk_token_id
        q_type = int(row.get('type', 0))

        return {
            "pixel_values": pixel_values,
            "input_ids": text_encoding['input_ids'].squeeze(0),
            "attention_mask": text_encoding['attention_mask'].squeeze(0),
            "labels": torch.tensor(label, dtype=torch.long),
            "q_type": torch.tensor(q_type, dtype=torch.long),
            "target_box": target_box,
            "has_box": has_box
        }

In [11]:
# ==========================================
# MODEL (Hỗ trợ Ablation: Structural + Depth)
# ==========================================

class CrossAttentionFusion(nn.Module):
    def __init__(self, visual_dim, text_dim, embed_dim, num_heads=8):
        super().__init__()
        self.vis_project = nn.Linear(visual_dim, embed_dim)
        self.txt_project = nn.Linear(text_dim, embed_dim)
        self.multihead_attn = nn.MultiheadAttention(embed_dim, num_heads, batch_first=True, dropout=0.1)
        self.norm = nn.LayerNorm(embed_dim)

    def forward(self, visual_features, text_features):
        v_proj = self.vis_project(visual_features)
        t_proj = self.txt_project(text_features)
        attn_output, _ = self.multihead_attn(query=t_proj, key=v_proj, value=v_proj)
        combined = self.norm(attn_output + t_proj)
        return combined.mean(dim=1) + combined.max(dim=1)[0]


class BasicBBoxHead(nn.Module):
    """BBox Head truyền thống: Chỉ nhìn ảnh, không quan tâm Text hay Type."""
    def __init__(self, visual_dim=768):
        super().__init__()
        self.regressor = nn.Sequential(
            nn.Linear(visual_dim, 512), nn.LayerNorm(512), nn.ReLU(),
            nn.Dropout(0.2), nn.Linear(512, 256), nn.ReLU(),
            nn.Linear(256, 4), nn.Sigmoid()
        )

    def forward(self, vis_seq):
        vis_pooled = vis_seq.mean(dim=1)
        return self.regressor(vis_pooled)


class TextGuidedBBoxHead(nn.Module):
    """BBox Head hướng dẫn bởi Text + Question Type."""
    def __init__(self, visual_dim=768, text_dim=768, type_dim=64):
        super().__init__()
        self.attn_proj = nn.Linear(text_dim + type_dim, visual_dim)
        self.vis_branch = nn.Sequential(
            nn.Linear(visual_dim, 512), nn.LayerNorm(512), nn.ReLU(),
            nn.Dropout(0.2), nn.Linear(512, 256), nn.ReLU(),
        )
        self.txt_branch = nn.Sequential(
            nn.Linear(text_dim + type_dim, 256), nn.LayerNorm(256), nn.ReLU(),
        )
        self.regressor = nn.Sequential(
            nn.Linear(512, 256), nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(256, 4), nn.Sigmoid()
        )

    def forward(self, vis_seq, text_feat, type_feat):
        context = torch.cat([text_feat, type_feat], dim=-1)
        attn_query = self.attn_proj(context).unsqueeze(-1)
        attn_weights = torch.bmm(vis_seq, attn_query).squeeze(-1)
        attn_weights = torch.softmax(attn_weights, dim=-1).unsqueeze(-1)
        attended_vis = (vis_seq * attn_weights).sum(dim=1)

        vis_feat = self.vis_branch(attended_vis)
        txt_feat = self.txt_branch(context)
        combined = torch.cat([vis_feat, txt_feat], dim=-1)
        return self.regressor(combined)


class ViVQATripleTaskModel(nn.Module):
    def __init__(self, num_classes, num_q_types, use_cross_attn=True, use_text_guided_box=True, qformer_unfreeze_layers=None):
        super().__init__()
        self.use_cross_attn = use_cross_attn
        self.use_text_guided_box = use_text_guided_box

        if qformer_unfreeze_layers is None:
            qformer_unfreeze_layers = CONFIG['qformer_unfreeze_layers']

        print(f"  Model | CrossAttn={use_cross_attn} | TextGuidedBox={use_text_guided_box} | QFormer Unfreeze={qformer_unfreeze_layers}")
        tmp_blip = Blip2Model.from_pretrained(CONFIG['blip_model'], torch_dtype=torch.float16)
        self.vision_model = tmp_blip.vision_model
        self.qformer = tmp_blip.qformer
        self.query_tokens = tmp_blip.query_tokens
        del tmp_blip

        for param in self.vision_model.parameters(): param.requires_grad = False
        for param in self.qformer.parameters(): param.requires_grad = False
        self.query_tokens.requires_grad = False
        self._unfreeze_qformer_last_layers(qformer_unfreeze_layers)

        self.phobert = AutoModel.from_pretrained(CONFIG['text_model'])

        embed_dim = 768
        if self.use_cross_attn:
            self.fusion = CrossAttentionFusion(visual_dim=768, text_dim=768, embed_dim=embed_dim)
        else:
            self.simple_fusion = nn.Linear(768 + 768, embed_dim)
            self.fusion_norm = nn.LayerNorm(embed_dim)

        self.q_type_head = nn.Linear(768, num_q_types)
        self.type_emb = nn.Embedding(num_q_types, CONFIG['type_embed_dim'])

        in_features = embed_dim + CONFIG['type_embed_dim']
        self.classifier = nn.Sequential(
            nn.Linear(in_features, 512), nn.LayerNorm(512), nn.GELU(),
            nn.Dropout(0.3), nn.Linear(512, num_classes)
        )

        if self.use_text_guided_box:
            self.bbox_head = TextGuidedBBoxHead(visual_dim=768, text_dim=768, type_dim=CONFIG['type_embed_dim'])
        else:
            self.bbox_head = BasicBBoxHead(visual_dim=768)

    def _unfreeze_qformer_last_layers(self, n_layers):
        try:
            layers = self.qformer.encoder.layer
            unfreeze_from = max(0, len(layers) - n_layers)
            for i, layer in enumerate(layers):
                if i >= unfreeze_from:
                    for param in layer.parameters(): param.requires_grad = True
            if n_layers > 0:
                self.query_tokens.requires_grad = True
        except AttributeError:
            pass

    def forward(self, pixel_values, input_ids, attention_mask):
        with torch.no_grad():
            vision_outputs = self.vision_model(pixel_values=pixel_values)
            image_embeds = vision_outputs.last_hidden_state
            image_attention_mask = torch.ones(image_embeds.size()[:-1], dtype=torch.long, device=image_embeds.device)

        query_tokens = self.query_tokens.expand(image_embeds.shape[0], -1, -1)
        image_embeds_d = image_embeds.detach()
        query_outputs = self.qformer(
            query_embeds=query_tokens, encoder_hidden_states=image_embeds_d,
            encoder_attention_mask=image_attention_mask, output_attentions=False
        )
        visual_seq = query_outputs.last_hidden_state

        text_outputs = self.phobert(input_ids=input_ids, attention_mask=attention_mask)
        text_feats = text_outputs.last_hidden_state
        visual_seq = visual_seq.to(text_feats.dtype)

        cls_token_feats = text_feats[:, 0, :]
        q_type_logits = self.q_type_head(cls_token_feats)
        soft_q_type = torch.softmax(q_type_logits, dim=-1)
        type_vec = torch.matmul(soft_q_type, self.type_emb.weight)

        if self.use_cross_attn:
            fused_vec = self.fusion(visual_seq, text_feats)
        else:
            vis_pooled = visual_seq.mean(dim=1)
            concat_feats = torch.cat([vis_pooled, cls_token_feats], dim=-1)
            fused_vec = self.fusion_norm(self.simple_fusion(concat_feats))

        final_features = torch.cat([fused_vec, type_vec], dim=-1)
        ans_logits = self.classifier(final_features)

        if self.use_text_guided_box:
            pred_boxes = self.bbox_head(visual_seq, cls_token_feats, type_vec)
        else:
            pred_boxes = self.bbox_head(visual_seq)

        return ans_logits, pred_boxes, q_type_logits

In [12]:
# ==========================================
# CHUẨN BỊ DEV SET (TẬP DỮ LIỆU THU NHỎ)
# ==========================================
def prepare_dev_data():
    blip_processor = Blip2Processor.from_pretrained(CONFIG['blip_model'])
    tokenizer = AutoTokenizer.from_pretrained(CONFIG['text_model'], use_fast=False)

    df_raw = pd.read_csv(CONFIG['train_csv'])
    if 'Unnamed: 0' in df_raw.columns: df_raw.rename(columns={'Unnamed: 0': 'question_id'}, inplace=True)
    df_box = pd.read_csv(CONFIG['train_box_csv'])
    if 'Unnamed: 0' in df_box.columns: df_box.rename(columns={'Unnamed: 0': 'question_id'}, inplace=True)

    df_merged = pd.merge(df_raw, df_box[['question_id', 'merged_box']], on='question_id', how='left')

    all_answers = df_merged['answer'].apply(lambda x: str(x).lower().strip()).unique().tolist()
    if 'unknown' not in all_answers: all_answers.append('unknown')
    label_encoder = LabelEncoder().fit(all_answers)
    unk_token_id = label_encoder.transform(['unknown'])[0]
    num_q_types = int(df_merged['type'].max() + 1)

    dev_df, val_df = train_test_split(
        df_merged, train_size=CONFIG['dev_set_ratio'], test_size=CONFIG['val_set_ratio'],
        random_state=42, stratify=df_merged['type']
    )
    print(f"Kho dữ liệu Ablation | Train Dev: {len(dev_df)} mẫu | Val Dev: {len(val_df)} mẫu | Q-Types: {num_q_types}")

    dev_loader = DataLoader(
        ViVQATripleTaskDataset(dev_df, CONFIG['train_img_dir'], blip_processor, tokenizer, label_encoder, unk_token_id),
        batch_size=CONFIG['batch_size'], shuffle=True, num_workers=2, pin_memory=True
    )
    val_loader = DataLoader(
        ViVQATripleTaskDataset(val_df, CONFIG['train_img_dir'], blip_processor, tokenizer, label_encoder, unk_token_id),
        batch_size=CONFIG['batch_size'], shuffle=False, num_workers=2, pin_memory=True
    )

    return dev_loader, val_loader, label_encoder, num_q_types, blip_processor, tokenizer, unk_token_id, val_df

In [13]:
# ==========================================
# INFERENCE CHIẾN LƯỢC THEO Q-TYPE
# ==========================================
def crop_and_reencode(image_pil, box_coords, blip_processor, min_size=32, pad_ratio=0.15):
    W, H = image_pil.size
    xmin, ymin, xmax, ymax = box_coords

    pad_x = (xmax - xmin) * pad_ratio
    pad_y = (ymax - ymin) * pad_ratio
    xmin, ymin = max(0.0, xmin - pad_x), max(0.0, ymin - pad_y)
    xmax, ymax = min(1.0, xmax + pad_x), min(1.0, ymax + pad_y)

    px0, py0 = int(xmin * W), int(ymin * H)
    px1, py1 = int(xmax * W), int(ymax * H)

    if (px1 - px0) < min_size or (py1 - py0) < min_size:
        cropped = image_pil
    else:
        cropped = image_pil.crop((px0, py0, px1, py1))

    return blip_processor(images=cropped, return_tensors="pt")['pixel_values']


def multi_strategy_inference_batch(model, batch, blip_processor, device, box_required_types=None, crop_weight=0.6, pad_ratio=0.15):
    """Inference multi-strategy: crop ảnh theo box cho câu hỏi thuộc box_required_types."""
    if box_required_types is None: box_required_types = CONFIG['box_required_types']

    pixel_values = batch['pixel_values'].to(device, dtype=torch.float16)
    input_ids, attention_mask = batch['input_ids'].to(device), batch['attention_mask'].to(device)

    with torch.no_grad(), autocast():
        ans_logits_full, pred_boxes, q_type_logits = model(pixel_values, input_ids, attention_mask)

    pred_qtypes = q_type_logits.argmax(1)
    ans_logits_final = ans_logits_full.clone()

    box_indices = [i for i, qt in enumerate(pred_qtypes.tolist()) if qt in box_required_types]

    if box_indices and 'raw_images' in batch:
        for i in box_indices:
            raw_img = batch['raw_images'][i]
            box_coords = pred_boxes[i].cpu().tolist()

            crop_pixels = crop_and_reencode(raw_img, box_coords, blip_processor, pad_ratio=pad_ratio).to(device, dtype=torch.float16)

            with torch.no_grad(), autocast():
                ans_logits_crop, _, _ = model(crop_pixels, input_ids[i:i+1], attention_mask[i:i+1])

            ans_logits_final[i] = (1 - crop_weight) * ans_logits_full[i] + crop_weight * ans_logits_crop.squeeze(0)

    pred_answers = ans_logits_final.argmax(1)
    return pred_answers, pred_boxes, pred_qtypes

In [14]:
# ==========================================
# PHASE 1: TRAINING LOSS WEIGHT ABLATION
# Grid search: lambda_bbox × lambda_giou
# ==========================================
def run_phase1_training_ablation():
    print("\n" + "="*60)
    print("  PHASE 1: TRAINING LOSS WEIGHT ABLATION")
    print("="*60)

    torch.manual_seed(42); np.random.seed(42)
    dev_loader, val_loader, label_encoder, num_q_types, _, _, _, _ = prepare_dev_data()

    keys, values = zip(*CONFIG['grid_training'].items())
    experiments = [dict(zip(keys, v)) for v in itertools.product(*values)]
    print(f"Tổng số tổ hợp: {len(experiments)}")

    results = []
    best_score = 0
    all_loss_curves = {}

    # Khởi tạo model 1 lần, reset weights mỗi experiment
    print("Khởi tạo model gốc (1 lần duy nhất)...")
    model = ViVQATripleTaskModel(len(label_encoder.classes_), num_q_types).to(CONFIG['device'])
    init_weights_path = "/content/vivqa_init_weights_temp.pth"
    torch.save(model.state_dict(), init_weights_path)

    for exp_idx, exp_config in enumerate(experiments):
        exp_name = f"bbox={exp_config['lambda_bbox']}_giou={exp_config['lambda_giou']}"
        print(f"\n[{exp_idx+1}/{len(experiments)}] {exp_name}")

        model.load_state_dict(torch.load(init_weights_path, weights_only=True))

        other_params = [p for n, p in model.named_parameters() if p.requires_grad and 'qformer' not in n and n != 'query_tokens']
        qformer_params = [p for p in model.qformer.parameters() if p.requires_grad]
        optimizer = torch.optim.AdamW([
            {'params': other_params, 'lr': 5e-5},
            {'params': qformer_params + [model.query_tokens], 'lr': 1e-5}
        ])

        criterion_vqa = nn.CrossEntropyLoss(label_smoothing=0.1)
        criterion_qtype = nn.CrossEntropyLoss()
        criterion_bbox = nn.SmoothL1Loss(reduction='none')
        scaler = GradScaler()

        epoch_losses = []

        for epoch in range(CONFIG['phase1_epochs']):
            model.train()
            epoch_loss_sum, epoch_steps = 0.0, 0

            for batch in tqdm(dev_loader, desc=f"Ep {epoch+1}", leave=False):
                pixel_values = batch['pixel_values'].to(CONFIG['device'], dtype=torch.float16)
                input_ids = batch['input_ids'].to(CONFIG['device'])
                attention_mask = batch['attention_mask'].to(CONFIG['device'])
                labels = batch['labels'].to(CONFIG['device'])
                q_types_gt = batch['q_type'].to(CONFIG['device'])
                target_box = batch['target_box'].to(CONFIG['device'])
                has_box = batch['has_box'].to(CONFIG['device'])

                optimizer.zero_grad()
                with autocast():
                    ans_logits, pred_boxes, q_type_logits = model(pixel_values, input_ids, attention_mask)
                    loss_vqa = criterion_vqa(ans_logits, labels)
                    loss_qtype = criterion_qtype(q_type_logits, q_types_gt)

                    target_types = torch.tensor(CONFIG['box_required_types'], device=CONFIG['device'])
                    color_mask = torch.isin(q_types_gt, target_types).float()
                    valid_box_mask = (has_box == 1.0) & (color_mask == 1.0)

                    loss_bbox_raw = criterion_bbox(pred_boxes, target_box).mean(dim=1)
                    loss_bbox = (loss_bbox_raw * valid_box_mask).sum() / (valid_box_mask.sum() + 1e-6)

                    if valid_box_mask.any():
                        loss_giou = giou_loss(pred_boxes[valid_box_mask].float(), target_box[valid_box_mask].float())
                    else:
                        loss_giou = torch.tensor(0.0, device=CONFIG['device'])

                    loss = (loss_vqa
                            + CONFIG['lambda_qtype'] * loss_qtype
                            + exp_config['lambda_bbox'] * loss_bbox
                            + exp_config['lambda_giou'] * loss_giou)

                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()

                epoch_loss_sum += loss.item()
                epoch_steps += 1

            epoch_losses.append(epoch_loss_sum / max(epoch_steps, 1))

        all_loss_curves[exp_name] = epoch_losses

        # === Đánh giá trên Val Dev (Extended Metrics) ===
        model.eval()
        all_preds, all_labels_list, all_qtypes_list, all_ious = [], [], [], []

        with torch.no_grad(), autocast():
            for batch in val_loader:
                pixel_values = batch['pixel_values'].to(CONFIG['device'], dtype=torch.float16)
                input_ids = batch['input_ids'].to(CONFIG['device'])
                attention_mask = batch['attention_mask'].to(CONFIG['device'])
                labels = batch['labels'].to(CONFIG['device'])
                target_box = batch['target_box'].to(CONFIG['device'])
                has_box = batch['has_box'].to(CONFIG['device'])
                q_types_gt = batch['q_type'].to(CONFIG['device'])

                ans_logits, pred_boxes, _ = model(pixel_values, input_ids, attention_mask)
                preds = ans_logits.argmax(1)
                all_preds.extend(preds.cpu().tolist())
                all_labels_list.extend(labels.cpu().tolist())
                all_qtypes_list.extend(q_types_gt.cpu().tolist())

                valid_mask = (has_box == 1.0)
                if valid_mask.any():
                    ious = calculate_iou(pred_boxes[valid_mask], target_box[valid_mask])
                    all_ious.extend(ious.cpu().tolist())

        metrics = compute_extended_metrics(all_preds, all_labels_list, all_qtypes_list, all_ious, num_q_types)
        combined_score = (metrics['vqa_acc'] + metrics.get('miou', 0) * 1.5) / 2.5

        print(f"   VQA={metrics['vqa_acc']:.4f} | F1-M={metrics['f1_macro']:.4f} | mIoU={metrics.get('miou', 0):.4f} | Score={combined_score:.4f}")

        if combined_score > best_score:
            best_score = combined_score
            torch.save(model.state_dict(), "/content/vivqa_phase1_temp_best.pth")

        result_row = {**exp_config, **metrics, 'combined_score': combined_score, 'final_train_loss': epoch_losses[-1]}
        results.append(result_row)

        del optimizer
        gc.collect(); torch.cuda.empty_cache()

    if os.path.exists(init_weights_path):
        os.remove(init_weights_path)

    df_results = pd.DataFrame(results)
    df_results.to_csv(CONFIG['phase1_result_csv'], index=False)

    best_row = df_results.loc[df_results['combined_score'].idxmax()]
    print(f"\n{'='*60}")
    print(f"BEST: lambda_bbox={best_row['lambda_bbox']}, lambda_giou={best_row['lambda_giou']}")
    print(f"Score={best_row['combined_score']:.4f} | VQA={best_row['vqa_acc']:.4f} | mIoU={best_row.get('miou', 0)}")
    print(f"{'='*60}")
    print(f"Đã lưu: {CONFIG['phase1_result_csv']}")

    return df_results, all_loss_curves

In [15]:
# ==========================================
# PHASE 2: INFERENCE STRATEGY ABLATION
# Grid search: pad_ratio × crop_weight (dùng model chính từ Bước 2)
# ==========================================
def run_phase2_inference_ablation():
    print("\n" + "="*60)
    print("  PHASE 2: INFERENCE STRATEGY ABLATION")
    print("="*60)

    if not os.path.exists(CONFIG['checkpoint_to_test']):
        print(f"LỖI: Không tìm thấy model tại: {CONFIG['checkpoint_to_test']}")
        print("Hãy chạy Bước 2 (train.py) trước để sinh model chính!")
        return None

    _, _, label_encoder, num_q_types, blip_processor, tokenizer, unk_token_id, val_df = prepare_dev_data()

    # Dataset đặc biệt cho Inference: cần raw_images để crop
    class ViVQAInferenceAblation(ViVQATripleTaskDataset):
        def __getitem__(self, idx):
            item = super().__getitem__(idx)
            img_id = self.data.iloc[idx].get('img_id', self.data.iloc[idx].get('image'))
            img_path = find_image_path(self.image_dir, img_id)
            try:
                item['raw_image'] = Image.open(img_path).convert("RGB") if img_path else Image.new('RGB', (224, 224))
            except Exception:
                item['raw_image'] = Image.new('RGB', (224, 224))
            return item

    def collate_fn(batch):
        raw_images = [item.pop('raw_image') for item in batch]
        collated = torch.utils.data.default_collate(batch)
        collated['raw_images'] = raw_images
        return collated

    infer_loader = DataLoader(
        ViVQAInferenceAblation(val_df, CONFIG['train_img_dir'], blip_processor, tokenizer, label_encoder, unk_token_id),
        batch_size=16, shuffle=False, num_workers=2, collate_fn=collate_fn
    )

    model = ViVQATripleTaskModel(len(label_encoder.classes_), num_q_types).to(CONFIG['device'])
    model.load_state_dict(torch.load(CONFIG['checkpoint_to_test'], map_location=CONFIG['device'], weights_only=True))
    model.eval()

    keys, values = zip(*CONFIG['grid_inference'].items())
    experiments = [dict(zip(keys, v)) for v in itertools.product(*values)]
    results = []

    for exp_idx, exp_config in enumerate(experiments):
        print(f"\n[{exp_idx+1}/{len(experiments)}] pad={exp_config['pad_ratio']}, crop_w={exp_config['crop_weight']}")

        all_preds, all_labels_list, all_qtypes_list = [], [], []

        with torch.no_grad():
            for batch in tqdm(infer_loader, leave=False):
                labels = batch['labels'].to(CONFIG['device'])
                q_types_gt = batch['q_type'].to(CONFIG['device'])

                pred_answers, _, _ = multi_strategy_inference_batch(
                    model, batch, blip_processor, CONFIG['device'],
                    box_required_types=CONFIG['box_required_types'],
                    crop_weight=exp_config['crop_weight'],
                    pad_ratio=exp_config['pad_ratio']
                )

                all_preds.extend(pred_answers.cpu().tolist())
                all_labels_list.extend(labels.cpu().tolist())
                all_qtypes_list.extend(q_types_gt.cpu().tolist())

        metrics = compute_extended_metrics(all_preds, all_labels_list, all_qtypes_list, num_q_types=num_q_types)

        print(f"   Overall={metrics['vqa_acc']:.4f} | F1-M={metrics['f1_macro']:.4f} | Type2={metrics.get('acc_type_2', 'N/A')}")

        result_row = {**exp_config, **metrics}
        results.append(result_row)

        gc.collect(); torch.cuda.empty_cache()

    df_results = pd.DataFrame(results)
    df_results.to_csv(CONFIG['phase2_result_csv'], index=False)

    best_row = df_results.loc[df_results['vqa_acc'].idxmax()]
    print(f"\nBEST: pad_ratio={best_row['pad_ratio']}, crop_weight={best_row['crop_weight']}")
    print(f"   VQA={best_row['vqa_acc']:.4f} | F1-M={best_row['f1_macro']:.4f}")
    print(f"Đã lưu: {CONFIG['phase2_result_csv']}")

    return df_results

In [16]:
# ==========================================
# PHASE 3: STRUCTURAL ABLATION (MODULE CONTRIBUTION)
# So sánh 4 biến thể kiến trúc, dùng lambda tốt nhất từ Phase 1
# ==========================================
def run_phase3_structural_ablation():
    print("\n" + "="*60)
    print("  PHASE 3: STRUCTURAL ABLATION (MODULE CONTRIBUTION)")
    print("="*60)

    torch.manual_seed(42); np.random.seed(42)
    dev_loader, val_loader, label_encoder, num_q_types, _, _, _, _ = prepare_dev_data()

    # Đọc lambda tốt nhất từ Phase 1 (hoặc dùng mặc định)
    best_lambda_bbox, best_lambda_giou = 2.0, 10.0
    if os.path.exists(CONFIG['phase1_result_csv']):
        df_p1 = pd.read_csv(CONFIG['phase1_result_csv'])
        best_row = df_p1.loc[df_p1['combined_score'].idxmax()]
        best_lambda_bbox = float(best_row['lambda_bbox'])
        best_lambda_giou = float(best_row['lambda_giou'])
        print(f"Lambda từ Phase 1: bbox={best_lambda_bbox}, giou={best_lambda_giou}")
    else:
        print(f"Chưa có Phase 1, dùng mặc định: bbox={best_lambda_bbox}, giou={best_lambda_giou}")

    structural_configs = [
        {"name": "1. Baseline (VQA Only, Concat)",
         "use_cross_attn": False, "use_text_guided_box": False,
         "lambda_bbox": 0.0, "lambda_qtype": 0.0, "lambda_giou": 0.0},
        {"name": "2. +Multi-task (Basic Box Head)",
         "use_cross_attn": False, "use_text_guided_box": False,
         "lambda_bbox": best_lambda_bbox, "lambda_qtype": CONFIG['lambda_qtype'], "lambda_giou": best_lambda_giou},
        {"name": "3. +CrossAttention Fusion",
         "use_cross_attn": True, "use_text_guided_box": False,
         "lambda_bbox": best_lambda_bbox, "lambda_qtype": CONFIG['lambda_qtype'], "lambda_giou": best_lambda_giou},
        {"name": "4. +TextGuided Box (Full Model)",
         "use_cross_attn": True, "use_text_guided_box": True,
         "lambda_bbox": best_lambda_bbox, "lambda_qtype": CONFIG['lambda_qtype'], "lambda_giou": best_lambda_giou},
    ]

    results = []

    for cfg in structural_configs:
        print(f"\n>>> {cfg['name']}")

        model = ViVQATripleTaskModel(
            len(label_encoder.classes_), num_q_types,
            use_cross_attn=cfg['use_cross_attn'],
            use_text_guided_box=cfg['use_text_guided_box']
        ).to(CONFIG['device'])

        trainable_params = count_trainable_params(model)
        print(f"   Trainable params: {format_number(trainable_params)}")

        other_params = [p for n, p in model.named_parameters() if p.requires_grad and 'qformer' not in n and n != 'query_tokens']
        qformer_params = [p for p in model.qformer.parameters() if p.requires_grad]
        optimizer = torch.optim.AdamW([
            {'params': other_params, 'lr': 5e-5},
            {'params': qformer_params + [model.query_tokens], 'lr': 1e-5}
        ])

        criterion_vqa = nn.CrossEntropyLoss(label_smoothing=0.1)
        criterion_qtype = nn.CrossEntropyLoss()
        criterion_bbox = nn.SmoothL1Loss(reduction='none')
        scaler = GradScaler()

        for epoch in range(CONFIG['phase3_epochs']):
            model.train()
            for batch in tqdm(dev_loader, desc=f"Ep {epoch+1}", leave=False):
                pixel_values = batch['pixel_values'].to(CONFIG['device'], dtype=torch.float16)
                input_ids = batch['input_ids'].to(CONFIG['device'])
                attention_mask = batch['attention_mask'].to(CONFIG['device'])
                labels = batch['labels'].to(CONFIG['device'])
                q_types_gt = batch['q_type'].to(CONFIG['device'])
                target_box = batch['target_box'].to(CONFIG['device'])
                has_box = batch['has_box'].to(CONFIG['device'])

                optimizer.zero_grad()
                with autocast():
                    ans_logits, pred_boxes, q_type_logits = model(pixel_values, input_ids, attention_mask)
                    loss = criterion_vqa(ans_logits, labels)

                    if cfg['lambda_bbox'] > 0:
                        loss_qtype = criterion_qtype(q_type_logits, q_types_gt)
                        target_types = torch.tensor(CONFIG['box_required_types'], device=CONFIG['device'])
                        color_mask = torch.isin(q_types_gt, target_types).float()
                        valid_box_mask = (has_box == 1.0) & (color_mask == 1.0)

                        loss_bbox_raw = criterion_bbox(pred_boxes, target_box).mean(dim=1)
                        loss_bbox = (loss_bbox_raw * valid_box_mask).sum() / (valid_box_mask.sum() + 1e-6)
                        loss_giou = giou_loss(pred_boxes[valid_box_mask].float(), target_box[valid_box_mask].float()) if valid_box_mask.any() else torch.tensor(0.0, device=CONFIG['device'])

                        loss = loss + cfg['lambda_qtype'] * loss_qtype + cfg['lambda_bbox'] * loss_bbox + cfg['lambda_giou'] * loss_giou

                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()

        # === Đánh giá ===
        model.eval()
        all_preds, all_labels_list, all_qtypes_list, all_ious = [], [], [], []

        with torch.no_grad(), autocast():
            for batch in val_loader:
                pixel_values = batch['pixel_values'].to(CONFIG['device'], dtype=torch.float16)
                input_ids = batch['input_ids'].to(CONFIG['device'])
                attention_mask = batch['attention_mask'].to(CONFIG['device'])
                labels = batch['labels'].to(CONFIG['device'])
                target_box = batch['target_box'].to(CONFIG['device'])
                has_box = batch['has_box'].to(CONFIG['device'])
                q_types_gt = batch['q_type'].to(CONFIG['device'])

                ans_logits, pred_boxes, _ = model(pixel_values, input_ids, attention_mask)
                preds = ans_logits.argmax(1)
                all_preds.extend(preds.cpu().tolist())
                all_labels_list.extend(labels.cpu().tolist())
                all_qtypes_list.extend(q_types_gt.cpu().tolist())

                valid_mask = (has_box == 1.0)
                if valid_mask.any() and cfg['lambda_bbox'] > 0:
                    ious = calculate_iou(pred_boxes[valid_mask], target_box[valid_mask])
                    all_ious.extend(ious.cpu().tolist())

        metrics = compute_extended_metrics(
            all_preds, all_labels_list, all_qtypes_list,
            all_ious if all_ious else None, num_q_types
        )

        print(f"   VQA={metrics['vqa_acc']:.4f} | F1-M={metrics['f1_macro']:.4f} | mIoU={metrics.get('miou', 'N/A')}")

        results.append({
            "Model Configuration": cfg['name'],
            "VQA Accuracy": metrics['vqa_acc'],
            "F1 Macro": metrics['f1_macro'],
            "F1 Weighted": metrics['f1_weighted'],
            "mIoU": metrics.get('miou', 'N/A'),
            "Box Acc@0.5": metrics.get('box_acc_05', 'N/A'),
            "Trainable Params": format_number(trainable_params),
        })

        del model, optimizer
        gc.collect(); torch.cuda.empty_cache()

    df_results = pd.DataFrame(results)
    df_results.to_csv(CONFIG['phase3_result_csv'], index=False)
    print(f"\nĐã lưu: {CONFIG['phase3_result_csv']}")
    return df_results

In [17]:
# ==========================================
# PHASE 4: QFORMER LAYER DEPTH ABLATION
# Giữ cố định lambda tốt nhất, thay đổi số layer QFormer unfreeze
# ==========================================
def run_phase4_depth_ablation():
    print("\n" + "="*60)
    print("  PHASE 4: QFORMER LAYER DEPTH ABLATION")
    print("="*60)

    torch.manual_seed(42); np.random.seed(42)
    dev_loader, val_loader, label_encoder, num_q_types, _, _, _, _ = prepare_dev_data()

    # Đọc lambda từ Phase 1
    best_lambda_bbox, best_lambda_giou = 2.0, 10.0
    if os.path.exists(CONFIG['phase1_result_csv']):
        df_p1 = pd.read_csv(CONFIG['phase1_result_csv'])
        best_row = df_p1.loc[df_p1['combined_score'].idxmax()]
        best_lambda_bbox = float(best_row['lambda_bbox'])
        best_lambda_giou = float(best_row['lambda_giou'])
        print(f"Lambda từ Phase 1: bbox={best_lambda_bbox}, giou={best_lambda_giou}")
    else:
        print(f"Chưa có Phase 1, dùng mặc định: bbox={best_lambda_bbox}, giou={best_lambda_giou}")

    results = []

    for n_layers in CONFIG['grid_qformer_layers']:
        print(f"\n>>> QFormer Unfreeze Layers: {n_layers}")

        model = ViVQATripleTaskModel(
            len(label_encoder.classes_), num_q_types,
            qformer_unfreeze_layers=n_layers
        ).to(CONFIG['device'])

        trainable_params = count_trainable_params(model)
        print(f"   Trainable params: {format_number(trainable_params)}")

        other_params = [p for n, p in model.named_parameters() if p.requires_grad and 'qformer' not in n and n != 'query_tokens']
        qformer_params = [p for p in model.qformer.parameters() if p.requires_grad]
        param_groups = [{'params': other_params, 'lr': 5e-5}]
        if qformer_params:
            param_groups.append({'params': qformer_params + [model.query_tokens], 'lr': 1e-5})
        elif model.query_tokens.requires_grad:
            param_groups.append({'params': [model.query_tokens], 'lr': 1e-5})
        optimizer = torch.optim.AdamW(param_groups)

        criterion_vqa = nn.CrossEntropyLoss(label_smoothing=0.1)
        criterion_qtype = nn.CrossEntropyLoss()
        criterion_bbox = nn.SmoothL1Loss(reduction='none')
        scaler = GradScaler()

        epoch_losses = []

        for epoch in range(CONFIG['phase4_epochs']):
            model.train()
            epoch_loss_sum, epoch_steps = 0.0, 0

            for batch in tqdm(dev_loader, desc=f"Depth={n_layers} Ep{epoch+1}", leave=False):
                pixel_values = batch['pixel_values'].to(CONFIG['device'], dtype=torch.float16)
                input_ids = batch['input_ids'].to(CONFIG['device'])
                attention_mask = batch['attention_mask'].to(CONFIG['device'])
                labels = batch['labels'].to(CONFIG['device'])
                q_types_gt = batch['q_type'].to(CONFIG['device'])
                target_box = batch['target_box'].to(CONFIG['device'])
                has_box = batch['has_box'].to(CONFIG['device'])

                optimizer.zero_grad()
                with autocast():
                    ans_logits, pred_boxes, q_type_logits = model(pixel_values, input_ids, attention_mask)
                    loss_vqa = criterion_vqa(ans_logits, labels)
                    loss_qtype = criterion_qtype(q_type_logits, q_types_gt)

                    target_types = torch.tensor(CONFIG['box_required_types'], device=CONFIG['device'])
                    color_mask = torch.isin(q_types_gt, target_types).float()
                    valid_box_mask = (has_box == 1.0) & (color_mask == 1.0)

                    loss_bbox_raw = criterion_bbox(pred_boxes, target_box).mean(dim=1)
                    loss_bbox = (loss_bbox_raw * valid_box_mask).sum() / (valid_box_mask.sum() + 1e-6)
                    loss_giou = giou_loss(pred_boxes[valid_box_mask].float(), target_box[valid_box_mask].float()) if valid_box_mask.any() else torch.tensor(0.0, device=CONFIG['device'])

                    loss = (loss_vqa
                            + CONFIG['lambda_qtype'] * loss_qtype
                            + best_lambda_bbox * loss_bbox
                            + best_lambda_giou * loss_giou)

                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()

                epoch_loss_sum += loss.item()
                epoch_steps += 1

            epoch_losses.append(epoch_loss_sum / max(epoch_steps, 1))

        # === Đánh giá ===
        model.eval()
        all_preds, all_labels_list, all_qtypes_list, all_ious = [], [], [], []

        with torch.no_grad(), autocast():
            for batch in val_loader:
                pixel_values = batch['pixel_values'].to(CONFIG['device'], dtype=torch.float16)
                input_ids = batch['input_ids'].to(CONFIG['device'])
                attention_mask = batch['attention_mask'].to(CONFIG['device'])
                labels = batch['labels'].to(CONFIG['device'])
                target_box = batch['target_box'].to(CONFIG['device'])
                has_box = batch['has_box'].to(CONFIG['device'])
                q_types_gt = batch['q_type'].to(CONFIG['device'])

                ans_logits, pred_boxes, _ = model(pixel_values, input_ids, attention_mask)
                preds = ans_logits.argmax(1)
                all_preds.extend(preds.cpu().tolist())
                all_labels_list.extend(labels.cpu().tolist())
                all_qtypes_list.extend(q_types_gt.cpu().tolist())

                valid_mask = (has_box == 1.0)
                if valid_mask.any():
                    ious = calculate_iou(pred_boxes[valid_mask], target_box[valid_mask])
                    all_ious.extend(ious.cpu().tolist())

        metrics = compute_extended_metrics(all_preds, all_labels_list, all_qtypes_list, all_ious, num_q_types)

        print(f"   VQA={metrics['vqa_acc']:.4f} | F1-M={metrics['f1_macro']:.4f} | mIoU={metrics.get('miou', 0):.4f}")

        results.append({
            'qformer_layers': n_layers,
            'trainable_params': trainable_params,
            'trainable_params_str': format_number(trainable_params),
            **metrics,
            'loss_curve': epoch_losses,
        })

        del model, optimizer
        gc.collect(); torch.cuda.empty_cache()

    df_results = pd.DataFrame(results)
    df_results.to_csv(CONFIG['phase4_result_csv'], index=False)

    best_row = df_results.loc[df_results['vqa_acc'].idxmax()]
    print(f"\nBEST: {int(best_row['qformer_layers'])} layers → VQA={best_row['vqa_acc']:.4f} | Params={best_row['trainable_params_str']}")
    print(f"Đã lưu: {CONFIG['phase4_result_csv']}")
    return df_results

In [18]:
# ==========================================
# VISUALIZATION & LATEX TABLE GENERATION
# ==========================================

def plot_phase1_heatmap(df):
    """Heatmap Phase 1: lambda_bbox × lambda_giou → Score."""
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    for idx, (metric, title) in enumerate([
        ('vqa_acc', 'VQA Accuracy'),
        ('miou', 'Mean IoU'),
        ('combined_score', 'Combined Score')
    ]):
        if metric not in df.columns:
            axes[idx].set_visible(False)
            continue
        pivot = df.pivot_table(values=metric, index='lambda_giou', columns='lambda_bbox', aggfunc='mean')
        sns.heatmap(pivot, annot=True, fmt='.4f', cmap='YlOrRd', ax=axes[idx], cbar_kws={'shrink': 0.8})
        axes[idx].set_title(title, fontsize=12, fontweight='bold')
        axes[idx].set_xlabel('λ_bbox')
        axes[idx].set_ylabel('λ_giou')
    plt.suptitle('Phase 1: Loss Weight Ablation', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f"{CONFIG['figures_dir']}/phase1_heatmap.png", dpi=300, bbox_inches='tight')
    plt.show()

def plot_phase1_loss_curves(all_loss_curves, top_n=6):
    """Đồ thị Training Loss convergence cho Phase 1."""
    fig, ax = plt.subplots(figsize=(10, 6))
    for name, losses in list(all_loss_curves.items())[:top_n]:
        ax.plot(range(1, len(losses)+1), losses, 'o-', label=name, linewidth=1.5, markersize=4)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Training Loss')
    ax.set_title('Phase 1: Training Loss Convergence', fontsize=12, fontweight='bold')
    ax.legend(fontsize=8, loc='upper right')
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(f"{CONFIG['figures_dir']}/phase1_loss_curves.png", dpi=300, bbox_inches='tight')
    plt.show()

def plot_phase2_heatmap(df):
    """Heatmap Phase 2: pad_ratio × crop_weight → Accuracy."""
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    for idx, (metric, title) in enumerate([
        ('vqa_acc', 'Overall VQA Accuracy'),
        ('acc_type_2', 'Color-Type Accuracy (Type 2)')
    ]):
        if metric not in df.columns:
            axes[idx].set_visible(False)
            continue
        pivot = df.pivot_table(values=metric, index='crop_weight', columns='pad_ratio', aggfunc='mean')
        sns.heatmap(pivot, annot=True, fmt='.4f', cmap='YlGnBu', ax=axes[idx], cbar_kws={'shrink': 0.8})
        axes[idx].set_title(title, fontsize=12, fontweight='bold')
        axes[idx].set_xlabel('pad_ratio')
        axes[idx].set_ylabel('crop_weight')
    plt.suptitle('Phase 2: Inference Strategy Ablation', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f"{CONFIG['figures_dir']}/phase2_heatmap.png", dpi=300, bbox_inches='tight')
    plt.show()

def plot_phase3_bar(df):
    """Bar Chart Phase 3: Structural Ablation."""
    fig, ax = plt.subplots(figsize=(12, 6))
    configs = df['Model Configuration'].tolist()
    x = np.arange(len(configs))
    width = 0.25

    bars1 = ax.bar(x - width, df['VQA Accuracy'].astype(float), width, label='VQA Accuracy', color='#2196F3')
    bars2 = ax.bar(x, df['F1 Macro'].astype(float), width, label='F1-Macro', color='#FF9800')
    miou_vals = pd.to_numeric(df['mIoU'], errors='coerce').fillna(0)
    bars3 = ax.bar(x + width, miou_vals, width, label='mIoU', color='#4CAF50')

    ax.set_ylabel('Score')
    ax.set_title('Phase 3: Structural Ablation - Module Contribution', fontsize=12, fontweight='bold')
    ax.set_xticks(x)
    short_names = [c.split('. ')[1] if '. ' in c else c for c in configs]
    ax.set_xticklabels(short_names, rotation=15, ha='right', fontsize=9)
    ax.legend()
    ax.set_ylim(0, 1.05)

    for bars in [bars1, bars2, bars3]:
        for bar in bars:
            h = bar.get_height()
            if h > 0:
                ax.annotate(f'{h:.3f}', xy=(bar.get_x() + bar.get_width()/2, h),
                           xytext=(0, 3), textcoords="offset points", ha='center', fontsize=7)
    plt.tight_layout()
    plt.savefig(f"{CONFIG['figures_dir']}/phase3_structural.png", dpi=300, bbox_inches='tight')
    plt.show()

def plot_phase4_line(df):
    """Line Chart Phase 4: QFormer Depth vs Performance."""
    fig, ax1 = plt.subplots(figsize=(10, 6))
    layers = df['qformer_layers'].tolist()

    line1, = ax1.plot(layers, df['vqa_acc'], 'o-', color='#2196F3', linewidth=2, markersize=8, label='VQA Accuracy')
    line2, = ax1.plot(layers, df['f1_macro'], 's--', color='#FF9800', linewidth=2, markersize=8, label='F1-Macro')
    lines = [line1, line2]

    if 'miou' in df.columns:
        miou_vals = pd.to_numeric(df['miou'], errors='coerce').fillna(0)
        line3, = ax1.plot(layers, miou_vals, '^-.', color='#4CAF50', linewidth=2, markersize=8, label='mIoU')
        lines.append(line3)

    ax1.set_xlabel('Số layer QFormer được mở khóa', fontsize=11)
    ax1.set_ylabel('Score', fontsize=11)
    ax1.set_xticks(layers)

    ax2 = ax1.twinx()
    line4, = ax2.plot(layers, df['trainable_params'], 'D:', color='#9C27B0', linewidth=2, markersize=8, label='Trainable Params')
    ax2.set_ylabel('Trainable Parameters', color='#9C27B0', fontsize=11)
    ax2.tick_params(axis='y', labelcolor='#9C27B0')
    lines.append(line4)

    ax1.legend(handles=lines, loc='center left', fontsize=9)
    ax1.set_title('Phase 4: QFormer Layer Depth Ablation', fontsize=12, fontweight='bold')
    ax1.grid(True, alpha=0.3)
    fig.tight_layout()
    plt.savefig(f"{CONFIG['figures_dir']}/phase4_depth.png", dpi=300, bbox_inches='tight')
    plt.show()

def generate_latex_table(df, caption, label):
    """Sinh LaTeX table tự động từ DataFrame."""
    latex = df.to_latex(index=False, float_format="%.4f", caption=caption, label=label)
    print(latex)
    return latex

def print_summary():
    """In tóm tắt tất cả kết quả có sẵn."""
    print("\n" + "="*60)
    print("  TỔNG HỢP KẾT QUẢ ABLATION STUDY")
    print("="*60)
    for name, path in [
        ("Phase 1", CONFIG['phase1_result_csv']),
        ("Phase 2", CONFIG['phase2_result_csv']),
        ("Phase 3", CONFIG['phase3_result_csv']),
        ("Phase 4", CONFIG['phase4_result_csv']),
    ]:
        if os.path.exists(path):
            df = pd.read_csv(path)
            print(f"\n{'='*40}")
            print(f"  {name} ({len(df)} experiments)")
            print(f"{'='*40}")
            print(df.to_markdown(index=False))
        else:
            print(f"\n  {name}: Chưa chạy (file không tồn tại)")

---
# BƯỚC 1: Chạy Phase 1 — Dò bộ trọng số λ tối ưu
- **Grid**: `lambda_bbox` × `lambda_giou` (16 tổ hợp × 10 epochs trên 15% data)
- **Đầu ra**: Bảng & Heatmap cho paper, file CSV trên Drive
- **Sau bước này**: Lấy bộ λ tốt nhất → quay về `train.py` cho Bước 2

In [19]:
# === CHẠY PHASE 1 ===
df_phase1, loss_curves = run_phase1_training_ablation()

# In bảng kết quả
print("\n[BẢNG 1] Loss Weight Ablation Results:")
cols = ['lambda_bbox', 'lambda_giou', 'vqa_acc', 'f1_macro', 'miou', 'combined_score']
display_cols = [c for c in cols if c in df_phase1.columns]
print(df_phase1[display_cols].sort_values('combined_score', ascending=False).to_markdown(index=False))

# Vẽ đồ thị
plot_phase1_heatmap(df_phase1)
plot_phase1_loss_curves(loss_curves)


  PHASE 1: TRAINING LOSS WEIGHT ABLATION


processor_config.json:   0%|          | 0.00/68.0 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/432 [00:00<?, ?B/s]

The image processor of type `BlipImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json: 0.00B [00:00, ?B/s]

...

   VQA=0.6183 | F1-M=0.3388 | mIoU=0.3805 | Score=0.4757

[2/16] bbox=0.5_giou=1.0


Ep 1:   0%|          | 0/57 [00:00<?, ?it/s]

Ep 2:   0%|          | 0/57 [00:00<?, ?it/s]

Ep 3:   0%|          | 0/57 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
import os
os._exit(0)

---
# BƯỚC 2: Quay về `train.py` — Train Full 100% Data
> **Bước này KHÔNG chạy ở đây.**
>
> 1. Mở file `train.py`
> 2. Điền bộ `lambda_bbox` và `lambda_giou` tốt nhất từ Phase 1 vào CONFIG
> 3. Train full 100% data → sinh file model chính (vd: `vivqa_v100_100percent.pth`)
> 4. Upload model lên Google Drive
> 5. Quay lại đây, cập nhật đường dẫn `CONFIG['checkpoint_to_test']` rồi chạy Phase 2

---
# BƯỚC 3: Chạy Phase 2 — Tìm chiến lược Inference tối ưu
- **Yêu cầu**: Phải có model chính từ Bước 2 tại `checkpoint_to_test`
- **Grid**: `pad_ratio` × `crop_weight` (30 tổ hợp, chỉ inference không train)
- **Đầu ra**: Bảng & Heatmap: pad × crop → Overall Acc & Color-Type Acc

In [ ]:
# === CHẠY PHASE 2 ===
df_phase2 = run_phase2_inference_ablation()

if df_phase2 is not None:
    print("\n[BẢNG 2] Inference Strategy Ablation Results:")
    cols = ['pad_ratio', 'crop_weight', 'vqa_acc', 'f1_macro', 'acc_type_2']
    display_cols = [c for c in cols if c in df_phase2.columns]
    print(df_phase2[display_cols].sort_values('vqa_acc', ascending=False).to_markdown(index=False))

    plot_phase2_heatmap(df_phase2)

---
# BƯỚC 4: Chạy Phase 3 — Chứng minh đóng góp của từng module
- **4 cấu hình**: Baseline → +Multi-task → +CrossAttn → +TextGuidedBox (Full)
- **Train từ đầu** trên subset 15%, dùng λ tốt nhất từ Phase 1
- **Đầu ra**: Bảng & Bar chart chứng minh mỗi module đều có giá trị

> Phase 3 chạy độc lập, không cần model từ Bước 2.

In [ ]:
# === CHẠY PHASE 3 ===
df_phase3 = run_phase3_structural_ablation()

print("\n[BẢNG 3] STRUCTURAL ABLATION — MODULE CONTRIBUTION:")
print(df_phase3.to_markdown(index=False))

plot_phase3_bar(df_phase3)

# Sinh LaTeX table cho paper
generate_latex_table(df_phase3, caption="Structural Ablation Results", label="tab:structural_ablation")

---
# BƯỚC 5: Chạy Phase 4 — Vision Encoder Depth Ablation
- **Grid**: Số layer QFormer unfreeze = [0, 1, 2, 3, 4, 6]
- **Giữ cố định** λ tốt nhất từ Phase 1, kiến trúc Full Model
- **Đầu ra**: Bảng + Line chart (Depth vs Performance & Params) cho paper
- **Mục đích**: Chứng minh mức fine-tune vision encoder tối ưu, tránh overfitting

> Dùng để vẽ đồ thị trade-off giữa "độ sâu fine-tune" và "hiệu suất".

In [ ]:
# === CHẠY PHASE 4 ===
df_phase4 = run_phase4_depth_ablation()

print("\n[BẢNG 4] QFORMER DEPTH ABLATION:")
cols = ['qformer_layers', 'trainable_params_str', 'vqa_acc', 'f1_macro', 'miou']
display_cols = [c for c in cols if c in df_phase4.columns]
print(df_phase4[display_cols].to_markdown(index=False))

plot_phase4_line(df_phase4)

# Sinh LaTeX table cho paper
generate_latex_table(
    df_phase4[display_cols],
    caption="QFormer Layer Depth Ablation",
    label="tab:depth_ablation"
)

---
# TỔNG HỢP KẾT QUẢ
In tất cả bảng kết quả và sinh LaTeX tables cho paper.

In [ ]:
# === TỔNG HỢP TẤT CẢ KẾT QUẢ ===
print_summary()

# Vẽ lại tất cả đồ thị (nếu file CSV đã có sẵn từ các lần chạy trước)
for name, path, plot_fn in [
    ("Phase 1", CONFIG['phase1_result_csv'], plot_phase1_heatmap),
    ("Phase 2", CONFIG['phase2_result_csv'], plot_phase2_heatmap),
    ("Phase 3", CONFIG['phase3_result_csv'], plot_phase3_bar),
    ("Phase 4", CONFIG['phase4_result_csv'], plot_phase4_line),
]:
    if os.path.exists(path):
        print(f"\n--- {name} ---")
        df = pd.read_csv(path)
        plot_fn(df)

print(f"\nTất cả đồ thị đã được lưu tại: {CONFIG['figures_dir']}")